In [245]:
import pandas as pd
import numpy as np
import re

In [246]:
emails_df = pd.read_csv('../data/02_cleaned_emails_unlabelled_internal.csv')
emails_df.head()

,Date,From,To,Subject,X_From,X_To,Message,DISC,Entire-Message
0,"Fri, 4 May 2001 13:51:00 -0700 (PDT)",phillip.allen@enron.com,john.lavorato@enron.com,Re:,Phillip K Allen,John J Lavorato <John J Lavorato/ENRON@enronXg...,Traveling to have a business meeting takes the...,NaN,Re: Traveling to have a business meeting takes...
1,"Mon, 23 Oct 2000 06:13:00 -0700 (PDT)",phillip.allen@enron.com,randall.gay@enron.com,No Subject,Phillip K Allen,Randall L Gay,"Randy,\n Can you send me a schedule of the sal...",NaN,"No Subject Randy,\n Can you send me a schedule..."
2,"Tue, 22 Aug 2000 07:44:00 -0700 (PDT)",phillip.allen@enron.com,"david.l.johnson@enron.com, john.shafer@enron.com",No Subject,Phillip K Allen,"david.l.johnson@enron.com, John Shafer",Please cc the following distribution list with...,NaN,No Subject Please cc the following distributio...
3,"Mon, 16 Oct 2000 06:42:00 -0700 (PDT)",phillip.allen@enron.com,buck.buckner@honeywell.com,Re: FW: fixed forward or other Collar floor ga...,Phillip K Allen,"""Buckner, Buck"" <buck.buckner@honeywell.com> @...","Mr. Buckner,\n For delivered gas behind San Di...",NaN,Re: FW: fixed forward or other Collar floor ga...
4,"Fri, 13 Oct 2000 06:45:00 -0700 (PDT)",phillip.allen@enron.com,stagecoachmama@hotmail.com,No Subject,Phillip K Allen,stagecoachmama@hotmail.com,"Lucy,\n Here are the rentrolls:\n Open them an...",NaN,"No Subject Lucy,\n Here are the rentrolls:\n O..."


In [247]:
emails_df.columns

Index(['Date', 'From', 'To', 'Subject', 'X_From', 'X_To', 'Message', 'DISC',
       'Entire-Message'],
      dtype='object')

In [248]:
emails_df_msg_label = emails_df.drop(columns=['X_From', 'X_To', 'Date', 'From', 'To', 'Subject', 'Message'])
emails_df_msg_label.head()

,DISC,Entire-Message
0,NaN,Re: Traveling to have a business meeting takes...
1,NaN,"No Subject Randy,\n Can you send me a schedule..."
2,NaN,No Subject Please cc the following distributio...
3,NaN,Re: FW: fixed forward or other Collar floor ga...
4,NaN,"No Subject Lucy,\n Here are the rentrolls:\n O..."


In [249]:
emails_df_rule_based = emails_df_msg_label.sample(5000, random_state=32).copy()
rule_based_ids = emails_df_rule_based.index.tolist()
print(rule_based_ids)

[34304, 29217, 30268, 32821, 10558, 18577, 26404, 63883, 1235, 26667, 7284, 58029, 15554, 17459, 58940, 30565, 37824, 56941, 34539, 57314, 15526, 25263, 30286, 63763, 29448, 64255, 62413, 47079, 48610, 3027, 13492, 58812, 32049, 21449, 45526, 28446, 56265, 46566, 16184, 14531, 10943, 40511, 15141, 27351, 3789, 37116, 36605, 18516, 27655, 46056, 51221, 4431, 21830, 26947, 34993, 61532, 4268, 62367, 28492, 16010, 41793, 8444, 6417, 62051, 27409, 35985, 18099, 14411, 62732, 33813, 54143, 23637, 8800, 15495, 36834, 52964, 27758, 10918, 15477, 4655, 16365, 29677, 11709, 64101, 15488, 41965, 13054, 28529, 24916, 14821, 60370, 55866, 30514, 8065, 45326, 21784, 61759, 55631, 25272, 48115, 36277, 44948, 11708, 37553, 10871, 26538, 17409, 52922, 40673, 60271, 46272, 25940, 63098, 51715, 5655, 26083, 33602, 12284, 39936, 63805, 4184, 4590, 11631, 63235, 12551, 5656, 64299, 13420, 40680, 46930, 39999, 15995, 61207, 64662, 25398, 36910, 5542, 39823, 31684, 63704, 8705, 44818, 55998, 22538, 27497, 9

In [250]:
# lowercasing and remove punctuation
PUNC_PATTERN = re.compile(r'[^\w\s]')

def preprocess_text_for_rule_based(text):
    if not isinstance(text, str):
        return ''


    text = PUNC_PATTERN.sub('', text)
    text = re.sub(r'\s+', ' ', text)
    text = text.strip().lower()
    return text

emails_df_rule_based['Entire-Message'] = emails_df_rule_based['Entire-Message'].apply(preprocess_text_for_rule_based)
emails_df_rule_based.head()

,DISC,Entire-Message
34304,NaN,no subject jeff can you set up a meetin with y...
29217,NaN,re tom costantino ill set him up on an intervi...
30268,NaN,thursday staff meetings urgent hi everyone wit...
32821,NaN,letter to stephen baum re solicitation of empl...
10558,NaN,filings there were 3 puc filings mentioned in ...


In [251]:
DISC_KEYWORDS = {
    'D' : ["deadline", "urgent", "now", "action", "priority", "resolve", "results",
        "decision", "goal", "challenge", "do it", "fast", "push", "lead", "firm", "pressure", "action required",
        "quickly", "at all costs", "make it happen", "efficiently", "asap"
        "take the lead", "take action", "do it now", "get this done", "make sure",
        "ensure", "enforce", "absolutely", "direct", "right now", "immediately",
        "speed", "I want", "I said", "must",  "put questions to rest", "your fault",
        "your mistake", "your apologies", "your bad", "your error", "your oversight",
        "your slip", "your blunder", "your carelessness", "your inattention",
        "your neglect", "your omission", "your oversight", "I guarantee",
        "enough is enough", " you need to"],
    'I' : ["excited", "great job", "lets", "share", "appreciate",
        "celebrate", "fun", "engage", "collaborate", "believe",
          "nice to hear", "amazing", "awesome", "fantastic", "get a hold of me",
          "discuss", "chat", "jokes", "laugh", "smile", "happy", "good news", "favorite",
          "favourite", "celebrate", "congratulations", "kudos", "celebration", "party",
          "playing", "exciting", "thrilling", "enjoy", "game"],
    'S' : ["kindly", "let me know", "available", "cooperate", "help", "understand",
        "follow up", "consistent", "appreciate your help", "thanks again",
        "dont worry", "help in any way I can", "let me know", "comfortable",
        "sorry for not responding", "thank you so much", "my fault", "my mistake",
        "my apologies", "my bad", "my error", "my oversight", "my slip", "my blunder",
        "my carelessness", "my inattention", "my neglect", "my omission", "my oversight",
        "sorry", "apologize for", "agree"],
    'C' : ["details", "data", "report", "analyze", "summary", "document", "proof",
        "schedule", "spreadsheet", "accuracy", "format", "process", "standards", "analysis",
        "compliance", "audit", "method", "evidence", "confirm", "structured", "sources",
        "review", "check", "validate", "verify", "validate", "as per", "according to",
        "in accordance with", "consistent with", "comply with", "follow", "procedure",
        "guideline", "manual", "policy", "regulation", "rule", "standard", "statute",
        "requirement", "specification", "protocol", "principle", "reconcile", "cross-check",
        "cross-reference", "corroborate", "substantiate", "authenticate", "confirm", "justify",
        "hindsight", "forethought", "foresight", "insight", "formula", "unjustified",
        "unsubstantiated", "unverified", "unvalidated", "unconfirmed", "unjust", "unreasonable",
        "looking into", "investigate", "examine", "probe", "scrutinize", "inspect", "identified",
        "final execution", "implementation", "calculation", "agreement", "calibrate"]
}

In [ ]:
# def match_disc_label(msg):
#     disc_labels = set()
#     for label, keywords in DISC_KEYWORDS.items():
#         for word in keywords:
#             if word in msg:
#                 disc_labels.add(label)

#     return disc_labels

def match_disc_label(msg, max_freq=2):
    labels_freq = {'D': 0, 'I': 0, 'S': 0, 'C': 0}
    for label, keywords in DISC_KEYWORDS.items():
        for word in keywords:
            if word in msg:
                labels_freq[label] += 1
    # print(labels_freq)

    filter_labels = {label: freq for label, freq in labels_freq.items() if freq > 0}
    if not filter_labels:
        return ()

    sorted_labels_freq = sorted(filter_labels.items(), key=lambda x: (-x[1], x[0]))
    top_labels = [label for label, freq in sorted_labels_freq[:max_freq]]
    return tuple(top_labels)

emails_df_rule_based['DISC_rule_based'] = emails_df_rule_based['Entire-Message'].apply(match_disc_label)

print(emails_df_rule_based.head(5))
print(emails_df_rule_based['DISC_rule_based'].value_counts())


       DISC                                     Entire-Message DISC_rule_based
34304   NaN  no subject jeff can you set up a meetin with y...              ()
29217   NaN  re tom costantino ill set him up on an intervi...          (C, D)
30268   NaN  thursday staff meetings urgent hi everyone wit...          (C, D)
32821   NaN  letter to stephen baum re solicitation of empl...              ()
10558   NaN  filings there were 3 puc filings mentioned in ...            (C,)
DISC_rule_based
(C, D)    1183
()         627
(C,)       551
(C, S)     429
(C, I)     340
(D,)       286
(S, D)     264
(S, C)     264
(I,)       218
(D, I)     196
(D, C)     191
(D, S)     167
(S,)       109
(I, C)      65
(I, D)      50
(I, S)      46
(S, I)      14
Name: count, dtype: int64


In [ ]:
# emails_df_rule_based_none = emails_df_rule_based[emails_df_rule_based['DISC_rule_based'].apply(lambda x: len(x) == 0)]
# print(emails_df_rule_based_none['Entire-Message'])

# emails_df_rule_based_none.to_csv('../data/msg_without_labels.csv', index=False)
# print(emails_df_rule_based_none.value_counts())

34304    no subject jeff can you set up a meetin with y...
32821    letter to stephen baum re solicitation of empl...
1235     wednesdays swap meeting canceled tomorrows swa...
37824    when you are ready hey guys call me when you a...
47079    map ben in regards to the chicago area we also...
                               ...                        
40285    baseball stats we talked a couple of weeks ago...
12341    board meeting rick i need your view on which o...
15061    pirate cake can have a cake with rough blue ic...
55780    the newshour with jim lehrer sponsorship oppor...
126      no subject daryl here is the file that include...
Name: Entire-Message, Length: 627, dtype: object
